# 03 · Attribution: transport, window and information

**RQ2:** Which mechanism changes forecast quality? The original practical rules
selected windows independently. This study fixes windows for mechanism comparisons,
keeps dimension and smoothing fixed, and changes transport or retained nodes.
No setting is selected using these exploratory evaluation scores.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from spde_covariance.empirical_research import load_research_evidence

evidence = load_research_evidence(Path("../experiments/results/shfe-research"))
print(evidence["protocol"]["status"])
print("Numerical source:", evidence["manifest"]["git_revision"])
print("Original baseline source:", evidence["protocol"]["baseline_source_revision"])
assert evidence["protocol"]["baseline_summary_sha256"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def table(rows, columns):
    print(" | ".join(columns))
    for row in rows:
        print(
            " | ".join(
                (
                    f"{row.get(c):.6g}" if isinstance(row.get(c), float) else str(row.get(c))
                    for c in columns
                )
            )
        )

## Protocol and fair comparisons

The adjusted factor model fits $C$ through $V_\Delta(C)$; its A=0 counterpart
uses ordinary coefficient increments. Both use identical reconstructed states,
window and future actual-contract outcomes. Compare raw direct covariance and
10% diagonal shrinkage at the same nominal windows. Actual historical support
can differ because raw current contracts have finite history.

Nested masks retain all nodes, alternating nodes, then four nodes from the
alternating set. They affect factor reconstruction only. Raw direct baselines
retain full contract history, so masked-factor versus direct comparisons are
information stress tests. Within-mask adjusted/A=0 comparisons isolate transport
more closely. A 16-interval history cannot meet the original 20-pair baseline
minimum; do not relax that minimum to manufacture a comparison.

In [ ]:
study = evidence["attribution"]
print(json.dumps(study["scope"], indent=2))
rows = [r for r in study["scores"] if r["mask"] == "all" and r["year"] == "all"]
table(rows, ["window", "exposure", "method", "count", "qlike", "risk_ratio"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, exposure in zip(axes, ["outright_3m", "spread_3m_6m", "butterfly_3m_6m_9m"], strict=True):
    for method, style in [
        ("factor_adjusted", "o-"),
        ("factor_unadjusted", "x--"),
        ("direct", "s-"),
        ("shrinkage", "^:"),
    ]:
        points = sorted(
            [r for r in rows if r["exposure"] == exposure and r["method"] == method],
            key=lambda r: r["window"],
        )
        ax.plot(
            [r["window"] for r in points],
            [r["qlike"] for r in points],
            style,
            label=method,
            linewidth=1,
        )
    ax.set(title=exposure, xlabel="Matched nominal window", ylabel="Mean QLIKE (lower is better)")
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()

## Paired transport effect and masking stress

Differences are calculated on each method pair's identical scored observations.
Negative means favor the adjusted model. Report block uncertainty and availability
alongside the mean. A very small adjusted/A=0 effect is not evidence that transport
is mathematically unnecessary; it limits the measured forecasting claim here.

In [ ]:
paired = [
    r
    for r in study["paired_comparisons"]
    if r["reference"] == "factor_unadjusted" and r["score"] == "qlike" and (r["year"] == "all")
]
table(
    paired,
    [
        "mask",
        "window",
        "exposure",
        "count",
        "mean",
        "bootstrap_standard_error",
        "bootstrap_percentile_95",
    ],
)
table(
    [r for r in study["availability"] if r["window"] == 16 and r["mask"] == "all"],
    ["window", "exposure", "method", "candidate_origins", "issued_count", "scored_count"],
)

In [ ]:
for row in study["representation"]:
    print(row["mask"], json.dumps(row["reconstruction"], indent=2))
table(
    [
        r
        for r in study["available_scores"]
        if r["window"] == 126 and r["year"] == "all" and (r["method"] == "factor_adjusted")
    ],
    ["mask", "exposure", "count", "qlike", "risk_ratio"],
)
print("Numerical diagnostics:")
for row in study["numerical_diagnostics"]:
    print(
        row["mask"],
        row["window"],
        row["unavailable_reasons"],
        row["relative_covariance_error_bound_max"],
    )

## Interpretation and next decision

Distinguish practical policy comparison from mechanism attribution. Neither a
masked curve model beating a baseline under one loss nor more coordinates alone
establishes an SPDE advantage. Use the joint-risk study to assess whether retained
residual information improves the economically relevant targets.

## Reproduction and research boundary

This notebook reads checksummed aggregate evidence, not raw exchange files.
The numerical algorithms are in `spde_covariance`; use the
[protocol](../docs/empirical-research-protocol.md) to rerun local calibration.
Execute this notebook with `uv run python scripts/check_notebook.py notebooks/03_performance_attribution.ipynb`.

The original historical benchmark remains fixed. New analyses on the inspected
2024–2025 period are exploratory, even though forecast issuance is past-only.
No final untouched confirmation period, executable trading backtest or physical
forcing identification is claimed. See the [roadmap](../README.md).